<a href="https://colab.research.google.com/github/ansh-stack00/launchpad/blob/main/lora_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  Step 1: Install Dependencies

In [1]:
!pip install -q \
  torch \
  transformers \
  datasets \
  peft \
  bitsandbytes \
  accelerate \
  trl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 49.1 MB/s eta 0:00:00


In [2]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "val.jsonl"
    }
)

dataset


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1080
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 120
    })
})

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [11]:
MAX_LENGTH = 512

def format_instruction(example):
    if example["input"].strip():
        prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
"""
    else:
        prompt = f"""### Instruction:
{example['instruction']}

### Response:
"""

    full_text = prompt + example["output"]

    tokenized = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    response_start = len(
        tokenizer(prompt, add_special_tokens=False)["input_ids"]
    )

    labels = [-100] * response_start + tokenized["input_ids"][response_start:]
    labels = labels[:MAX_LENGTH]

    tokenized["labels"] = labels
    return tokenized



In [12]:
tokenized_ds = dataset.map(
    format_instruction,
    remove_columns=dataset["train"].column_names,
    batched=False
)


Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

In [14]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [15]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


## 🔹 Step 8: Configure LoRA

In [16]:
model.gradient_checkpointing_enable()


In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=500,
    eval_strategy="steps",
    save_strategy="epoch",
    eval_steps=500,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)


## 🚀 Step 10: Initialize Trainer

In [19]:
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
500,1.157374,1.283609


TrainOutput(global_step=810, training_loss=1.3413290388790178, metrics={'train_runtime': 973.5789, 'train_samples_per_second': 3.328, 'train_steps_per_second': 0.832, 'total_flos': 1.031921417060352e+16, 'train_loss': 1.3413290388790178, 'epoch': 3.0})

## 🎓 Step 11: Train the Model

In [20]:
adapter_path = "./adapters"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

('./adapters/tokenizer_config.json',
 './adapters/chat_template.jinja',
 './adapters/tokenizer.json')

In [20]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config
)

model = PeftModel.from_pretrained(base_model, adapter_path)
prompt = """### Instruction:
Answer the medical question accurately.

### Input:
What is the outlook for Lip and Oral Cavity Cancer ?
### Response:
"""
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = decoded.split("### Response:")[-1].strip()

print("\n" + "=" * 80)
print("MODEL RESPONSE")
print("=" * 80)
print(response)
print("=" * 80)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


MODEL RESPONSE
Certain factors affect prognosis (chance of recovery) and treatment options. The prognosis (chance of recovery) and treatment options depend on the following:         - Whether the cancer is early stage or advanced.    - Whether the tumor has metastasized to other parts of the body, including bones and lymph nodes.    - Whether the cancer is located in a region where the risk of recurrence is high.        Treatment options may include the following:     - Radiation therapy.    - Chemotherapy.    - Hormonal therapy.    - Surgery.    - Immunotherapy.    - Targeted therapy.    - Radiosensitization


In [21]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "./adapters"
MERGED_PATH = "./quantized/fp16-merged"

In [23]:
from peft import PeftModel
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.merge_and_unload()

model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./quantized/fp16-merged/tokenizer_config.json',
 './quantized/fp16-merged/chat_template.jinja',
 './quantized/fp16-merged/tokenizer.json')

In [24]:
from transformers import BitsAndBytesConfig

int8_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_int8 = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    device_map="auto",
    quantization_config=int8_config
)

model_int8.save_pretrained("./quantized/model-int8")
tokenizer.save_pretrained("./quantized/model-int8")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./quantized/model-int8/tokenizer_config.json',
 './quantized/model-int8/chat_template.jinja',
 './quantized/model-int8/tokenizer.json')

In [25]:
int4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    device_map="auto",
    quantization_config=int4_config
)

model_int4.save_pretrained("./quantized/model-int4")
tokenizer.save_pretrained("./quantized/model-int4")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./quantized/model-int4/tokenizer_config.json',
 './quantized/model-int4/chat_template.jinja',
 './quantized/model-int4/tokenizer.json')

In [26]:
!python llama.cpp/convert_hf_to_gguf.py \
  ./quantized/fp16-merged \
  --outfile ./quantized/model-fp16.gguf


INFO:hf-to-gguf:Loading model: fp16-merged
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:heuristics detected float16 tensor dtype, setting --outtype f16
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         

In [27]:
./build/bin/quantize \
  ../quantized/model.gguf \
  ../quantized/model-q4_0.gguf \
  q4_0


SyntaxError: invalid syntax (ipython-input-2305490973.py, line 1)

In [29]:
! llama.cpp/build/bin/llama-quantize \
  ./quantized/model-fp16.gguf \
  ./quantized/model.gguf \
  q4_0

main: build = 7971 (5fa1c190d)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing './quantized/model-fp16.gguf' to './quantized/model.gguf' as Q4_0
llama_model_loader: loaded meta data with 31 key-value pairs and 201 tensors from ./quantized/model-fp16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Fp16 Merged
llama_model_loader: - kv   3:                           general.finetune str              = merged
llama_model_loader: - kv   4:                         general.size_label str              = 1.1B
llama_model_loader: - kv   5:                          llama.block_count u32              = 22
llama_model_loa

In [32]:
! llama.cpp/build/bin/llama-cli \
  -m ./quantized/model.gguf \
  -p "Explain gradient checkpointing in simple terms." \
  -n 128



Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/- 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b7971-5fa1c190d
model      : model.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> Explain gradient checkpointing in simple terms.

|-\|/-\|/-\ In computer programming, gradient checkpointing is a technique for quickly restoring a program's execution state to a specific point in time during a run. It works by tracking the changes made by the program to its data structure, or state, as the program progresses.

The key idea behind gradient checkpointing is that the 

In [36]:
from datasets import load_dataset

val_ds = load_dataset(
    "json",
    data_files="val.jsonl"
)["train"]

sample = val_ds[1]

instruction = sample["instruction"]
input_text = sample["input"]


Generating train split: 0 examples [00:00, ? examples/s]

In [37]:
def build_prompt(instruction, input_text):
    if input_text.strip():
        return f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
"""
    else:
        return f"""### Instruction:
{instruction}

### Response:
"""


In [38]:

prompt = build_prompt(instruction, input_text)
print(prompt)


### Instruction:
Extract the requested medical information from the text.

### Input:
Do you have information about Tornadoes

### Response:



In [39]:
import time
import torch

def measure_speed(model, tokenizer, prompt, max_new_tokens=100, runs=3):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # warmup
    with torch.no_grad():
        model.generate(**inputs, max_new_tokens=20)

    times = []
    for _ in range(runs):
        start = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=max_new_tokens)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    return max_new_tokens / avg_time  # tokens/sec


In [44]:
# FP16 merged model
model_fp16 = AutoModelForCausalLM.from_pretrained(
    "./quantized/fp16-merged",
    dtype=torch.float16,
    device_map="auto"
)

# INT8 model
model_int8 = AutoModelForCausalLM.from_pretrained(
    "./quantized/model-int8",
    device_map="auto"
)

# INT4 model
model_int4 = AutoModelForCausalLM.from_pretrained(
    "./quantized/model-int4",
    device_map="auto"
)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [46]:
speed_fp16 = measure_speed(model_fp16, tokenizer, prompt)
speed_int8 = measure_speed(model_int8, tokenizer, prompt)
speed_int4 = measure_speed(model_int4, tokenizer, prompt)


print("FP16:", speed_fp16)
print("INT8:", speed_int8)
print("INT4:", speed_int4)



FP16: 25.179718672357783
INT8: 9.496983752564612
INT4: 18.452687553090527


In [53]:
! llama.cpp/./build/bin/llama-cli \
  -m quantized/model.gguf \
  -p f"{prompt}"\
  -n 100


Loading model... |-\|/-\|/-\|/-\|/-\|/ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b7971-5fa1c190d
model      : model.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> f### Instruction:
Explain gradient checkpointing in simple terms.

### Response:

|-\|/-\|/-\|/-\|/-\|/-\|/- So, when we think of restoring a model to a particular point in time, we usually do this by saving a snapshot of the model's parameters or state. Now, what happens in the background when we save these parameters or states is called a checkpoint. The checkpoint is like a snapshot, but it

In [54]:
!pip install torch transformers peft accelerate psutil llama-cpp-python sentence-transformers pandas -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [55]:
GGUF_MODEL = "./quantized/model.gguf"
PROMPTS = [
    """### Instruction:
Answer the medical question accurately.

### Input:
What are the treatments for Heart Attack ?

### Response:
""",

    """### Instruction:
Answer the medical question accurately.

### Input:
What is (are) Low Vision ?

### Response:
""",

    """### Instruction:
Answer the medical question accurately.

### Input:
Is Ovarian Epithelial, Fallopian Tube, and Primary Peritoneal Cancer inherited ?

### Response:
"""
]


GROUND_TRUTH = [
    "Heart attack treatment focuses on quickly restoring blood flow using thrombolytic drugs or angioplasty, followed by cardiac rehabilitation, lifestyle changes, and medications to prevent further damage.",

    "People with low vision can receive support services such as vision rehabilitation, counseling, recreation programs, and job training through community and state agencies for the visually impaired.",

    "About 20% of ovarian, fallopian tube, and primary peritoneal cancers are caused by inherited gene mutations, often associated with breast or colon cancer, and genetic testing is recommended for high-risk families."
]


In [57]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [58]:
def get_vram():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0

def accuracy(preds, refs):
    p_emb = embedder.encode(preds, convert_to_tensor=True)
    r_emb = embedder.encode(refs, convert_to_tensor=True)

    sims = util.cos_sim(p_emb, r_emb)

    per_sample_scores = sims.diag().cpu().numpy()

    # for i, score in enumerate(per_sample_scores):
    #     print(f"Sample {i+1} Similarity: {score:.3f}")

    mean_score = per_sample_scores.mean()

    return mean_score

In [61]:
import time
import torch
import psutil
import pandas as pd
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer, util

In [64]:


def benchmark_gguf(label):
    llm = Llama(model_path=GGUF_MODEL, n_ctx=2048, n_threads=8, verbose=False)

    outputs = []
    start = time.time()

    for p in PROMPTS:
        response = ""
        stream = llm(p, max_tokens=256, stream=True)

        for output in stream:
            token = output["choices"][0]["text"]
            response += token
            # print(token, end="", flush=True)

        # print("\n \n")
        outputs.append(response)

    end = time.time()

    tokens = sum(len(o.split()) for o in outputs)
    tps = tokens / (end - start)
    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps, 2),
        "Latency(s)": round(end - start, 2),
        "VRAM(MB)": 0,
        "Accuracy": round(acc, 3)
    }

results = []

results.append(benchmark_gguf("GGUF Q4 llama.cpp"))

In [65]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer

import threading

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
FT_MODEL = "./quantized/fp16-merged"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_PATH = "results.csv"



def benchmark_hf(model_path, label):

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map=DEVICE)

    outputs = []
    start = time.time()

    for prompt in PROMPTS:
        streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

        generation_kwargs = dict(
            **inputs,
            max_new_tokens=256,
            streamer=streamer,
            pad_token_id=tokenizer.eos_token_id
        )

        thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()

        response = ""
        for token in streamer:
            response += token
        #     print(token, end="", flush=True)

        # print("\n \n")
        outputs.append(response)
        thread.join()

    end = time.time()

    total_tokens = sum(len(tokenizer.encode(r)) for r in outputs)
    duration = end - start
    tps = total_tokens / duration
    acc = accuracy(outputs, GROUND_TRUTH)

    return {
        "Model": label,
        "Tokens/sec": round(tps, 2),
        "Latency(s)": round(duration, 2),
        "VRAM(MB)": round(get_vram(), 2),
        "Accuracy": round(acc, 3)
    }


results.append(benchmark_hf(BASE_MODEL, "Base Model"))
results.append(benchmark_hf(FT_MODEL, "Fine-tuned"))

df = pd.DataFrame(results)
df.to_csv(RESULTS_PATH, index=False)

print(df)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

               Model  Tokens/sec  Latency(s)  VRAM(MB)  Accuracy
0  GGUF Q4 llama.cpp        6.21       90.96      0.00     0.531
1         Base Model       40.31       12.58   9197.46     0.630
2         Fine-tuned       36.26       24.63   9197.46     0.702


In [66]:
import json

path = "lora_train.ipynb"

with open(path, "r") as f:
    nb = json.load(f)

nb["metadata"].pop("widgets", None)

with open(path, "w") as f:
    json.dump(nb, f, indent=1)


FileNotFoundError: [Errno 2] No such file or directory: 'lora_train.ipynb'